# Tile-level classification

This notebook walks through soma's **tile-level dataset path** — the one
behind patch-classification benchmarks like EVA — end to end:

```
TileDataset  ->  FeatureExtractor  ->  train (head)  ->  evaluate
```

Here every sample is **one already-cropped tile image with a class label**,
not a whole-slide image. That changes the shape of the pipeline versus the
[slide-level MIL walkthrough](walkthrough-slide-mil.ipynb):

* **No tissue masking or tiling.** Your inputs are already tiles, so there is
  no `PreprocessingConfig` — soma reads each `image_path` directly.
* **No MIL aggregator.** A tile encoder maps each tile to a single vector, so
  the task head consumes that vector as-is (`aggregator=None`). There is no
  bag to pool.

What stays the same is soma's core property: features don't depend on the
labels, so a **binary** and a **multiclass** head are just a different
`TaskConfig` on the **same extracted features**. The last cell collapses the
whole thing into a single `Pipeline` call.

> **This uses a tiny synthetic dataset so it runs anywhere on CPU with no
> gated model and no real slides.** The numbers are therefore meaningless —
> the point is the API, not the result.

## ⚠️ Scaffolding (not soma API)

The cell below fabricates a toy dataset: a handful of small RGB tile images
plus the two CSVs soma expects. **You would replace this with your own tiles
and labels** — the only thing that matters downstream is the on-disk
contract, which is identical to the slide-level one:

* `dataset.csv` — one row per **tile** with `sample_id`, `image_path`, `label`
  (extra columns are kept as free metadata; you pick which one is `label`).
  The only difference from the slide path is that `image_path` points at an
  individual tile image, not a whole slide.
* `splits.csv` — `sample_id`, `split` (`train` / `tune` / `test*`), optional
  `fold`.

In [ ]:
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

WORK = Path(tempfile.mkdtemp(prefix='soma-tile-tutorial-'))
TILES = WORK / 'tiles'; TILES.mkdir()
rng = np.random.default_rng(0)

def make_toy_tile(path, label, size=224):
    """A 224x224 RGB tile whose color tint is a weak, noisy signal for the
    label — enough for the API to run, not enough to mean anything."""
    base = np.array([[150, 70, 160], [70, 150, 90]][label % 2], np.int16)
    img = base + rng.integers(-40, 40, (size, size, 3))
    Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)).save(path)

N = 8
sample_ids = [f't{i:02d}' for i in range(N)]

# train/tune/test assignment (single fold) with both classes in every split
split = (['train'] * 4) + (['tune'] * 2) + (['test'] * 2)
binary = [0, 1, 0, 1,  0, 1,  0, 1]

for sid, y in zip(sample_ids, binary):
    make_toy_tile(TILES / f'{sid}.png', y)

dataset_csv = WORK / 'dataset.csv'
splits_csv = WORK / 'splits.csv'
pd.DataFrame({'sample_id': sample_ids,
              'image_path': [str(TILES / f'{s}.png') for s in sample_ids],
              'label': binary}).to_csv(dataset_csv, index=False)
pd.DataFrame({'sample_id': sample_ids, 'split': split}).to_csv(splits_csv, index=False)

print('dataset.csv'); print(pd.read_csv(dataset_csv).head().to_string(index=False))
print('\nsplits.csv'); print(pd.read_csv(splits_csv)['split'].value_counts().to_string())

## 1. Load the dataset and splits

`Dataset` reads `dataset.csv` and infers the label space; `Splits` reads
`splits.csv` and pairs it with the dataset. This is identical to every other
path — soma never partitions your data, so the splits you provide are the
splits it uses, which is what keeps evaluation reproducible and leakage-free.

In [ ]:
from soma import TileDataset, Splits

dataset = TileDataset(dataset_csv)
splits = Splits(splits_csv, dataset)
print('tiles:   ', len(dataset.sample_ids))
print('classes: ', sorted(pd.read_csv(dataset_csv)['label'].unique()))
print('folds:   ', splits.num_folds)

## 2. Encode each tile to a vector

This is where the tile path diverges from slide-level. There is no tissue
masking and no tiling — the inputs are already tiles — so the explicit
**`TileDataset`** type selects the tile path in `FeatureExtractor`. We pass no
`PreprocessingConfig`. It loads each `image_path`, applies the encoder's
transform, and writes **one 1-D vector per tile**.

We use [phikon](https://huggingface.co/owkin/phikon) because it is
**ungated** (no HF token) and small enough to run on CPU. With
`CacheConfig(enabled=True)` the vectors are cached so the second task below
reuses this one extraction. The result's `source` is the `FeatureStore`.

In [ ]:
from soma import FeatureExtractor, EncoderConfig, CacheConfig

extractor = FeatureExtractor(
    dataset,
    EncoderConfig(name='phikon'),
    cache=CacheConfig(enabled=True, root_dir=str(WORK / 'cache')),
    output_root=str(WORK / 'output'),
)
store = extractor.extract().source

vec = store.load(store.available_samples[0])
print('encoded', len(store.available_samples), 'tiles')
print('one vector per tile:', store.is_slide_level, '| shape', tuple(vec.shape))

## 3. Train a classifier head directly (no aggregator)

`train()` consumes the `FeatureStore` and trains a **task head** straight on
the per-tile vectors. Because each sample is already one vector, we pass
`aggregator=None` and `dataset_type='tile'` — there is no bag to pool.
`TaskConfig(name='binary_classification')` picks the head, loss, and metrics.

In [ ]:
from soma import TaskConfig, TrainingConfig, EvalConfig, train

training = TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4, seed=0)

result = train(
    feature_store=store,
    dataset=dataset,
    splits=splits,
    aggregator=None,          # <- tile features need no aggregator
    dataset_type='tile',
    task=TaskConfig(name='binary_classification'),
    training=training,
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    run_dir=str(WORK / 'runs' / 'binary'),
)
print('run dir:', result.run_dir)

## 4. Swap the task — same features, new head

Features don't depend on the labels, so we **reuse the same `FeatureStore`**
and only change the `Dataset` labels + `TaskConfig` + metrics. No
re-extraction — the cached vectors from step 2 are reused as-is.

In [ ]:
def relabel(values):
    df = pd.DataFrame({'sample_id': sample_ids,
                       'image_path': [str(TILES / f'{s}.png') for s in sample_ids],
                       'label': values})
    path = WORK / f'dataset_{abs(hash(tuple(map(str, values)))) % 10**6}.csv'
    df.to_csv(path, index=False)
    return TileDataset(path)

multiclass = [0, 1, 2, 0,  1, 2,  0, 1]
ds_mc = relabel(multiclass)
result_mc = train(
    feature_store=store, dataset=ds_mc, splits=Splits(splits_csv, ds_mc),
    aggregator=None, dataset_type='tile',
    task=TaskConfig(name='multiclass_classification'),
    training=training, evaluation=EvalConfig(metrics=['balanced_accuracy', 'accuracy']),
    run_dir=str(WORK / 'runs' / 'multiclass'),
)
print('multiclass run dir:', result_mc.run_dir)

## 5. The one-shot `Pipeline` equivalent

Everything above — encode, train, evaluate — is what `Pipeline` does in a
single call from one config. Note what is **absent** versus the slide-level
config: no `preprocessing` block and no `aggregator`. Setting
`dataset_type='tile'` is what routes the pipeline through
the canonical `FeatureExtractor` and the single-vector head.

*(Shown for reference, not executed — it would repeat the work above.)*

```python
from soma import (
    Pipeline, PipelineConfig, EncoderConfig,
    TaskConfig, TrainingConfig, EvalConfig, CacheConfig,
)

config = PipelineConfig(
    dataset_csv=str(dataset_csv),
    splits_csv=str(splits_csv),
    output_root='output/tile-binary',
    dataset_type='tile',       # <- constructs TileDataset; no aggregator
    encoder=EncoderConfig(name='phikon'),
    task=TaskConfig(name='binary_classification'),
    training=TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4),
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    cache=CacheConfig(enabled=True),
)
results = Pipeline(config).run()
```